# Neural Network Month · Project 1
## Notebook 3 of 3 — Does It Generalise?

One validation split can flatter or punish a model by luck. **5-fold cross-validation** repeats the experiment five times, each time holding out a different fifth of the learners, so every learner gets graded exactly once by a network that never saw them.

**This notebook:** 5-fold validation, a comparison against a plain linear baseline, predicted-versus-actual plots, and "what if" sleep curves. Three figures.

## 1 · Setup, shared code, data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# One palette for every figure in the series
NEURO_PALETTE = {
    "input":    "#7A9E7E",  # sage   - input neurons and features
    "hidden":   "#C97B4A",  # clay   - hidden neurons
    "recall":   "#2F6F73",  # teal   - recall target
    "reaction": "#D9A441",  # ochre  - reaction-time target
    "ink":      "#4A3F35",  # bark   - text, seed data, baselines
    "paper":    "#F5EFE6",  # sand   - background
    "grid":     "#D8CFC0",  # stone  - gridlines
}
P = NEURO_PALETTE
NEURO_CMAP = LinearSegmentedColormap.from_list("neuro", [P["recall"], P["paper"], P["hidden"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

In [ ]:
FEATURES = ["sleep_hrs", "study_hrs", "stress"]
TARGETS  = ["recall_score", "reaction_ms"]

def true_process(sleep, study, stress, rng=None):
    """The hidden 'biology' our network must rediscover. Noise only if rng is given."""
    sleep_norm  = np.clip((sleep - 4) / 4, 0, 1)
    sleep_gain  = 25 * (1 - np.exp(-(sleep - 4) / 2))                     # saturating
    study_gain  = 22 * (1 - np.exp(-study / 2)) * (0.5 + 0.5 * sleep_norm) # diminishing + needs sleep
    stress_cost = 0.35 * stress ** 1.5                                     # accelerating
    recall   = 35 + sleep_gain + study_gain - stress_cost
    reaction = 250 + 90 * np.exp(-(sleep - 4) / 2.5) * (1 + 0.1 * (stress - 1)) + 3 * (stress - 1) + 4 * np.maximum(study - 4, 0)
    if rng is not None:
        recall   = recall   + rng.normal(0, 3,  size=np.shape(recall))
        reaction = reaction + rng.normal(0, 8,  size=np.shape(reaction))
    return np.clip(recall, 0, 100), reaction

def make_seed(n, rng):
    """A small 'real-looking' cohort of learners."""
    sleep  = np.clip(rng.normal(6.8, 1.1, n), 4, 9.5)
    study  = rng.uniform(0.5, 6, n)
    stress = np.clip(rng.normal(5, 2, n), 1, 10)
    recall, reaction = true_process(sleep, study, stress, rng)
    return pd.DataFrame({"seed_id": np.arange(n), "sleep_hrs": sleep, "study_hrs": study,
                         "stress": stress, "recall_score": recall, "reaction_ms": reaction})

def smooth_bootstrap(seed_df, n_total, rng, jitter=0.05):
    """Resample whole rows with replacement, then nudge each value by 5% of its spread."""
    cols = FEATURES + TARGETS
    idx = rng.integers(0, len(seed_df), n_total)
    boot = seed_df.iloc[idx].reset_index(drop=True).copy()
    sd = seed_df[cols].std().values
    boot[cols] = boot[cols].values + rng.normal(0, 1, (n_total, len(cols))) * sd * jitter
    boot["sleep_hrs"]    = boot["sleep_hrs"].clip(4, 9.5)
    boot["study_hrs"]    = boot["study_hrs"].clip(0.5, 6)
    boot["stress"]       = boot["stress"].clip(1, 10)
    boot["recall_score"] = boot["recall_score"].clip(0, 100)
    return boot

In [ ]:
def assign_folds(seed_ids, k=5, rng=None):
    """Assign each SEED learner to a fold, so bootstrap copies of one person never straddle train and test."""
    rng = rng or np.random.default_rng(7)
    unique = rng.permutation(np.unique(seed_ids))
    fold_of = {sid: i % k for i, sid in enumerate(unique)}
    return np.array([fold_of[s] for s in seed_ids])

In [ ]:
class TinyNet:
    """3 inputs -> 6 tanh hidden neurons -> 2 linear outputs, trained by backpropagation."""
    def __init__(self, n_in=3, n_hidden=6, n_out=2, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden));      self.b1 = np.zeros(n_hidden)
        self.W2 = r.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, n_out)); self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.X  = X
        self.Z1 = X @ self.W1 + self.b1      # weighted sum into each hidden neuron
        self.A1 = np.tanh(self.Z1)           # hidden neurons "fire"
        self.out = self.A1 @ self.W2 + self.b2
        return self.out

    def backward(self, Y, lr):
        n, n_out = Y.shape
        d_out = 2 * (self.out - Y) / (n * n_out)      # d(loss)/d(output)
        dW2 = self.A1.T @ d_out;  db2 = d_out.sum(0)
        dA1 = d_out @ self.W2.T                       # blame passed back to hidden layer
        dZ1 = dA1 * (1 - self.A1 ** 2)                # tanh derivative
        dW1 = self.X.T @ dZ1;     db1 = dZ1.sum(0)
        self.W1 -= lr * dW1; self.b1 -= lr * db1
        self.W2 -= lr * dW2; self.b2 -= lr * db2

def mse(pred, Y): return float(np.mean((pred - Y) ** 2))

def r2_per_target(pred, Y):
    return 1 - ((Y - pred) ** 2).sum(0) / ((Y - Y.mean(0)) ** 2).sum(0)

def fit_scaler(A): return A.mean(0), A.std(0)

def train(net, Xtr, Ytr, Xva=None, Yva=None, epochs=3000, lr=0.3):
    hist = {"train": [], "val": []}
    for _ in range(epochs):
        pred = net.forward(Xtr)
        hist["train"].append(mse(pred, Ytr))
        if Xva is not None:
            hist["val"].append(mse(net.forward(Xva), Yva))
            net.forward(Xtr)                      # restore cached activations for backprop
        net.backward(Ytr, lr)
    return hist

In [ ]:
# Loads the cohort from Notebook 1; rebuilds it identically if the file is missing
if Path("neuro_cohort.csv").exists():
    cohort = pd.read_csv("neuro_cohort.csv")
    seed = pd.read_csv("neuro_seed.csv")
else:
    rng = np.random.default_rng(42)
    seed = make_seed(300, rng)
    cohort = smooth_bootstrap(seed, 1500, rng)
print(f"Seed learners: {len(seed)} | Bootstrapped rows: {len(cohort)}")
cohort.head()

## 2 · Five folds, no leakage

Folds are assigned by **seed learner**, so bootstrap copies of the same person never appear in both training and testing. Each fold gets its own scaler fit on its own training rows, a fresh network, and a plain linear regression as a baseline. If the neural network cannot beat a straight line, the hidden layer is not earning its place.

We also compute a **noise ceiling**: the score you would get by knowing the true hidden rule exactly. Because we added random noise, even a perfect model cannot reach R² = 1, so this tells us how much headroom is really there.

In [ ]:
fold = assign_folds(cohort["seed_id"].values, k=5)
X = cohort[FEATURES].values; Y = cohort[TARGETS].values

oof_pred = np.zeros_like(Y)
r2_nn, r2_lin, r2_cap, models, scalers = [], [], [], [], []

for k in range(5):
    tr, te = fold != k, fold == k
    x_mu, x_sd = fit_scaler(X[tr]); y_mu, y_sd = fit_scaler(Y[tr])
    Xtr, Xte = (X[tr] - x_mu) / x_sd, (X[te] - x_mu) / x_sd
    Ytr = (Y[tr] - y_mu) / y_sd

    net = TinyNet(seed=k)
    train(net, Xtr, Ytr, epochs=3000, lr=0.3)
    pred = net.forward(Xte) * y_sd + y_mu
    oof_pred[te] = pred
    r2_nn.append(r2_per_target(pred, Y[te]))
    models.append(net); scalers.append((x_mu, x_sd, y_mu, y_sd))

    # linear baseline: least squares with an intercept
    A = np.c_[Xtr, np.ones(len(Xtr))]
    coef, *_ = np.linalg.lstsq(A, Ytr, rcond=None)
    pred_lin = (np.c_[Xte, np.ones(len(Xte))] @ coef) * y_sd + y_mu
    r2_lin.append(r2_per_target(pred_lin, Y[te]))

    # noise ceiling: score of the TRUE hidden rule on this fold (no model could do better on average)
    rule = np.c_[true_process(X[te, 0], X[te, 1], X[te, 2])[0], true_process(X[te, 0], X[te, 1], X[te, 2])[1]]
    r2_cap.append(r2_per_target(rule, Y[te]))
    print(f"Fold {k+1}: NN R2 recall {r2_nn[-1][0]:.3f}, reaction {r2_nn[-1][1]:.3f} | "
          f"linear recall {r2_lin[-1][0]:.3f}, reaction {r2_lin[-1][1]:.3f}")

r2_nn, r2_lin, r2_cap = np.array(r2_nn), np.array(r2_lin), np.array(r2_cap)

## 3 · Figure 8 — fold-by-fold scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(5); w = 0.36
ax.bar(x - w/2, r2_nn[:, 0], w, color=P["recall"], label="Recall (network)")
ax.bar(x + w/2, r2_nn[:, 1], w, color=P["reaction"], label="Reaction (network)")
ax.scatter(x - w/2, r2_lin[:, 0], marker="D", s=45, color=P["ink"], zorder=3, label="Linear baseline")
ax.scatter(x + w/2, r2_lin[:, 1], marker="D", s=45, color=P["ink"], zorder=3)
ax.scatter(x - w/2, r2_cap[:, 0], marker="_", s=500, linewidths=2.5, color=P["ink"], zorder=3, label="Noise ceiling")
ax.scatter(x + w/2, r2_cap[:, 1], marker="_", s=500, linewidths=2.5, color=P["ink"], zorder=3)
lo = min(r2_nn.min(), r2_lin.min())
ax.set_ylim(max(0, lo - 0.1), 1.0)
ax.set_xticks(x); ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylabel("R-squared (higher is better)")
ax.set_title("Five-Fold Validation Scores"); ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1), fontsize=10)
fig.tight_layout(); save_fig(fig, "08_fold_scores"); plt.show()

summary = pd.DataFrame({
    "Network mean R2": r2_nn.mean(0), "Network std": r2_nn.std(0),
    "Linear mean R2": r2_lin.mean(0), "Linear std": r2_lin.std(0),
    "Ceiling mean R2": r2_cap.mean(0)}, index=["Recall", "Reaction"]).round(3)
summary

Read this honestly. The bars are the network, the diamonds are the straight-line baseline, and the dashes are the best any model could do given our noise. In our run the network averages ahead of the linear model on both outcomes and lands within a few hundredths of the ceiling, but the edge is **modest** and it does not win every fold (the linear model takes recall in fold 5 and reaction in fold 2; your numbers may shift slightly). The relationships in this data are only mildly nonlinear, so a simple model is a fair competitor. Networks pull far ahead when the structure is complicated, and saying so is part of understanding the technique.

Also note why we report a ceiling: R-squared depends on how spread out the outcomes are in each fold. A fold that happens to hold similar learners gives every model a lower score, even the true rule.

## 4 · Figure 9 — predicted versus actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.9))
for j, (ax, c, title, unit) in enumerate(zip(axes, [P["recall"], P["reaction"]],
                                              ["Recall Score", "Reaction Time"], ["score", "ms"])):
    ax.scatter(Y[:, j], oof_pred[:, j], s=10, alpha=0.35, color=c)
    lims = [min(Y[:, j].min(), oof_pred[:, j].min()), max(Y[:, j].max(), oof_pred[:, j].max())]
    ax.plot(lims, lims, color=P["ink"], linewidth=2, linestyle="--", label="Perfect prediction")
    ax.set_xlabel(f"Actual ({unit})"); ax.set_ylabel(f"Predicted ({unit})")
    ax.set_title(title); ax.legend(frameon=False, loc="upper left")
fig.suptitle("Predicted Versus Actual", y=1.02)
fig.tight_layout(); save_fig(fig, "09_pred_vs_actual"); plt.show()

Every point here is an **out-of-fold** prediction: made by a network that never saw that learner (or the learner's bootstrap siblings). Scatter around the dashed line is mostly the irreducible noise we built into the data.

## 5 · Figure 10 — a "what if" for sleep

Hold study at 3 hours and stress at 5, then sweep sleep from 4 to 9.5 hours. The solid line is the noiseless hidden rule. The thin lines are our five fold-networks, and the bold line is their average. If the networks truly learned the mechanism, they should trace the curve, including its flattening.

In [ ]:
grid = np.linspace(4, 9.5, 100)
Xg = np.c_[grid, np.full_like(grid, 3.0), np.full_like(grid, 5.0)]
true_recall, true_react = true_process(grid, np.full_like(grid, 3.0), np.full_like(grid, 5.0))

preds = []
for net, (x_mu, x_sd, y_mu, y_sd) in zip(models, scalers):
    preds.append(net.forward((Xg - x_mu) / x_sd) * y_sd + y_mu)
preds = np.array(preds)                     # (5 folds, 100, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for j, (ax, c, truth, title, yl) in enumerate(zip(axes, [P["recall"], P["reaction"]], [true_recall, true_react],
                                                  ["Recall Response", "Reaction Response"],
                                                  ["Recall score", "Reaction time (ms)"])):
    for f in range(5):
        ax.plot(grid, preds[f, :, j], color=c, alpha=0.35, linewidth=1.2)
    ax.plot(grid, preds[:, :, j].mean(0), color=c, linewidth=3, label="Network average")
    ax.plot(grid, truth, color=P["ink"], linewidth=2, linestyle="--", label="Hidden rule")
    ax.set_xlabel("Sleep hours"); ax.set_ylabel(yl); ax.set_title(title); ax.legend(frameon=False)
fig.suptitle("Learned Sleep Response", y=1.02)
fig.tight_layout(); save_fig(fig, "10_sleep_response"); plt.show()

The networks track the hidden rule closely through the middle of the range, and the five folds agree. Look at the edges, though: below roughly 5 hours or above roughly 8.5 hours the thin lines fan out. Few learners in our cohort sleep that little or that much, so the networks have little evidence there. **A network is only as trustworthy as the data under it, and it will still give you a confident-looking curve where the data runs out.**

## What we showed

- A tiny network (3 → 6 → 2) can predict **two outcomes at once** from a shared hidden layer.
- **Backpropagation** is just the chain rule, applied layer by layer, driving gradient descent.
- **5-fold cross-validation**, with folds grouped by seed learner, gives an honest estimate. Naive splitting on bootstrapped data would have leaked.
- The data is synthetic, so a good score demonstrates the *method*, not a fact about sleepy humans.

**Coming up this month:** how network design choices shape what is learned, and a closing project that puts real neurological data next to these ideas.

In [ ]:
# Optional: download this notebook's figures
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb3_figures", "zip", "figures")
    files.download("nb3_figures.zip")
except ImportError:
    print("Figures are saved in the 'figures' folder.")